# Amazon Review Rating Prediction with DistilBERT

**Task**: Given a raw Amazon review text, predict its star rating (1 ★ or 5 ★) by fine-tuning DistilBERT as a regression head.

**Dataset**: [Amazon Reviews (FastText format)](https://www.kaggle.com/datasets/bittlingmayer/amazonreviews) — 3.6 M binary-labelled reviews.

**Model**: `distilbert-base-uncased` → [CLS] → Dropout → Linear(768, 1)

| Stage | Details |
|---|---|
| Framework | PyTorch + Hugging Face Transformers |
| Training | Mixed-precision (AMP), linear warm-up + decay, early stopping |
| Evaluation | MAE · RMSE · Rounded-star accuracy · F1 |

---

## 1 · Environment Check

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")

## 2 · Package Installation

In [ ]:
# Install all dependencies in a single call to avoid repeated solver runs
!pip install -q kagglehub torch torchvision torchaudio transformers \
               tqdm seaborn scikit-learn

In [ ]:
import kagglehub

path = kagglehub.dataset_download("bittlingmayer/amazonreviews")
print("Dataset path:", path)

## 3 · Imports & Global Configuration

All third-party imports are declared once at the top.  
`Config` is a frozen dataclass that acts as a single source of truth — hyperparameters
that were previously scattered as magic numbers (e.g. `patience = 2`) now live here.

In [ ]:
import bz2
import os
import random
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")  # global plot theme


@dataclass
class Config:
    # ── Model ──────────────────────────────────────────────────────────
    model_name:   str   = "distilbert-base-uncased"
    dropout:      float = 0.3

    # ── Tokeniser ──────────────────────────────────────────────────────
    max_len:      int   = 128

    # ── Training ───────────────────────────────────────────────────────
    batch_size:   int   = 32
    lr:           float = 2e-5
    num_epochs:   int   = 20
    warmup_ratio: float = 0.1
    patience:     int   = 3          # early-stopping patience
    grad_clip:    float = 1.0

    # ── Data / I/O ─────────────────────────────────────────────────────
    seed:         int   = 42
    num_workers:  int   = 4          # DataLoader parallel workers
    checkpoint:   str   = "best_distilbert_regressor.pt"


cfg = Config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def set_seed(seed: int = 42) -> None:
    """Pin every random source for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)

## 4 · Data Loading

The Amazon Reviews dataset ships as a FastText-format `.bz2` archive.  
Each line: `__label__1 <text>` (1-star) or `__label__2 <text>` (5-star).

We pool train + test splits, shuffle, then carve out our own 70/15/15 split
so the ratio is explicit and reproducible.

In [ ]:
TRAIN_PATH = "/kaggle/input/amazonreviews/train.ft.txt.bz2"
TEST_PATH  = "/kaggle/input/amazonreviews/test.ft.txt.bz2"

LABEL_MAP = {"__label__1": 1.0, "__label__2": 5.0}


def load_fasttext_bz2(path: str, max_samples: int = None) -> pd.DataFrame:
    """
    Parse a FastText-format bz2 file into a DataFrame.

    Parameters
    ----------
    path        : Path to the .bz2 file.
    max_samples : Optional hard cap on rows read (for memory budgeting).

    Returns
    -------
    pd.DataFrame with columns: review_text (str), rating (float).
    """
    texts, ratings = [], []

    with bz2.open(path, mode="rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples is not None and i >= max_samples:
                break
            line = line.strip()
            if not line:
                continue
            try:
                label, text = line.split(" ", 1)
            except ValueError:
                continue                       # malformed line — skip silently
            if label not in LABEL_MAP:
                continue
            texts.append(text)
            ratings.append(LABEL_MAP[label])

    return pd.DataFrame({"review_text": texts, "rating": ratings})


# ── Load raw data ─────────────────────────────────────────────────────────
train_raw = load_fasttext_bz2(TRAIN_PATH, max_samples=300_000)
test_raw  = load_fasttext_bz2(TEST_PATH,  max_samples=100_000)

# ── Pool → shuffle → filter very short reviews ───────────────────────────
df = (
    pd.concat([train_raw, test_raw], ignore_index=True)
      .sample(n=300_000, random_state=cfg.seed)
      .reset_index(drop=True)
)
df = df[df["review_text"].str.split().str.len() >= 3].reset_index(drop=True)

print(f"Total samples : {len(df):,}")
print(f"\nRating counts :")
print(df["rating"].value_counts().rename({1.0: "1★ (neg)", 5.0: "5★ (pos)"}).to_string())

## 5 · Exploratory Data Analysis

Before modelling we check:
1. **Class balance** — whether negative and positive reviews are roughly equal.
2. **Review length** — how many tokens the tokeniser will need to handle (informs `max_len`).
3. **Sample reviews** — a sanity check on label quality.

In [ ]:
df["word_count"] = df["review_text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Rating distribution ──────────────────────────────────────────────────
counts = df["rating"].value_counts().sort_index()
bars   = axes[0].bar(
    ["1 ★  Negative", "5 ★  Positive"], counts.values,
    color=["#e74c3c", "#2ecc71"], width=0.45, edgecolor="white", linewidth=0.8,
)
axes[0].set_title("Rating Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Reviews")
axes[0].set_ylim(0, counts.max() * 1.15)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1_500,
        f"{val:,}\n({val / len(df) * 100:.1f} %)",
        ha="center", va="bottom", fontsize=11,
    )

# ── Word-count distribution ───────────────────────────────────────────────
med = df["word_count"].median()
axes[1].hist(df["word_count"].clip(upper=200), bins=60,
             color="steelblue", edgecolor="white", linewidth=0.4)
axes[1].axvline(med, color="crimson", linestyle="--", linewidth=1.8,
                label=f"Median = {med:.0f} words")
axes[1].axvline(cfg.max_len, color="darkorange", linestyle=":", linewidth=1.8,
                label=f"max_len = {cfg.max_len}")
axes[1].set_title("Review Length Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Word Count (clipped at 200)")
axes[1].set_ylabel("Number of Reviews")
axes[1].legend()

fig.suptitle("Amazon Reviews — Dataset Overview", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Sample reviews ────────────────────────────────────────────────────────
for label, rating in [("Negative (1 ★)", 1.0), ("Positive (5 ★)", 5.0)]:
    print(f"\n── {label} ──")
    for txt in df.query("rating == @rating")["review_text"].sample(2, random_state=7).values:
        print(f"  • {txt[:120]}{'...' if len(txt) > 120 else ''}")

## 6 · Data Splits & PyTorch Dataset

We use a deterministic 70 / 15 / 15 split (train / val / test).  
`ReviewDataset` tokenises lazily inside `__getitem__` — this keeps RAM usage flat
regardless of dataset size because token tensors are created on demand, not pre-computed.

**DataLoader improvements over the original:**
- `num_workers=4` — parallel data loading overlaps CPU tokenisation with GPU training.
- `pin_memory=True` — on GPU, tensors are pinned in host RAM for faster `to(device)` transfers.

In [ ]:
def train_val_test_split(
    dataframe: pd.DataFrame,
    val_size:  float = 0.15,
    test_size: float = 0.15,
    seed:      int   = 42,
) -> tuple:
    """
    Shuffle-once, then carve test → val → train in index order.
    Using a fixed random_state guarantees the same split every run.
    """
    df  = dataframe.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n   = len(df)
    n_t = int(n * test_size)
    n_v = int(n * val_size)

    return (
        df.iloc[n_t + n_v :].reset_index(drop=True),   # train
        df.iloc[n_t : n_t + n_v].reset_index(drop=True),  # val
        df.iloc[: n_t].reset_index(drop=True),          # test
    )


train_df, val_df, test_df = train_val_test_split(
    df, val_size=0.15, test_size=0.15, seed=cfg.seed
)

for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dist = split_df["rating"].value_counts().to_dict()
    print(f"{split_name:5s} : {len(split_df):>7,}  | {dist}")

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(cfg.model_name)


class ReviewDataset(Dataset):
    """
    Map-style dataset wrapping (review_text, rating) pairs.

    Tokenisation is deferred to __getitem__ (lazy) to avoid materialising
    the full token matrix in memory before training begins.
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int) -> None:
        self.texts     = df["review_text"].astype(str).tolist()
        self.ratings   = df["rating"].astype("float32").values
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),       # (max_len,)
            "attention_mask": enc["attention_mask"].squeeze(0),  # (max_len,)
            "rating":         torch.tensor(self.ratings[idx], dtype=torch.float),
        }


_pin = torch.cuda.is_available()  # pin memory only when a GPU is present

train_dataset = ReviewDataset(train_df, tokenizer, cfg.max_len)
val_dataset   = ReviewDataset(val_df,   tokenizer, cfg.max_len)
test_dataset  = ReviewDataset(test_df,  tokenizer, cfg.max_len)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=_pin)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=_pin)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=_pin)

print(f"Train : {len(train_dataset):>7,} samples  ({len(train_loader):,} batches)")
print(f"Val   : {len(val_dataset):>7,} samples  ({len(val_loader):,} batches)")
print(f"Test  : {len(test_dataset):>7,} samples  ({len(test_loader):,} batches)")

## 7 · Model Architecture

```
Input tokens (max_len=128)
        │
  DistilBERT encoder  (6 transformer layers, 66 M params)
        │
  [CLS] hidden state  (768-dim)
        │
  Dropout(p=0.3)
        │
  Linear(768 → 1)
        │
  Scalar rating ∈ ℝ
```

The `[CLS]` token's final hidden state serves as the sequence-level representation,
consistent with standard BERT-family fine-tuning practice.  
We add `weight_decay=0.01` to AdamW (L2 regularisation) which was absent in the original.

In [ ]:
class DistilBertRegressor(nn.Module):
    """
    DistilBERT encoder with a single-neuron regression head.

    The [CLS] token representation (index 0 of the final hidden state) is
    passed through dropout and a linear layer to produce a scalar rating.
    """

    def __init__(self, model_name: str, dropout: float = 0.3) -> None:
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.regressor  = nn.Linear(self.distilbert.config.hidden_size, 1)

    def forward(
        self,
        input_ids:      torch.Tensor,   # (B, L)
        attention_mask: torch.Tensor,   # (B, L)
    ) -> torch.Tensor:                  # (B,)
        out       = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr  = out.last_hidden_state[:, 0, :]         # (B, 768)
        return self.regressor(self.dropout(cls_repr)).squeeze(1)  # (B,)


# ── Instantiate ──────────────────────────────────────────────────────────
model = DistilBertRegressor(cfg.model_name, dropout=cfg.dropout).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

# ── Loss ─────────────────────────────────────────────────────────────────
criterion = nn.MSELoss()

# ── Optimiser (added weight_decay for L2 regularisation) ─────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)

# ── Scheduler: linear warmup → linear decay ───────────────────────────────
total_steps = len(train_loader) * cfg.num_epochs
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(cfg.warmup_ratio * total_steps),
    num_training_steps=total_steps,
)

# ── Mixed-precision scaler (GPU only) ─────────────────────────────────────
scaler = torch.amp.GradScaler(device_type="cuda") if torch.cuda.is_available() else None
print(f"\nAMP GradScaler : {'enabled' if scaler else 'disabled (CPU mode)'}")

## 8 · Training Utilities

Key improvements over the original:

| Change | Reason |
|---|---|
| `scaler` passed as parameter (not global) | Makes `run_epoch` self-contained and unit-testable |
| `tqdm` progress bar on every batch | Visibility during long epochs |
| `set_to_none=True` in `zero_grad` | Slightly faster than zeroing; avoids unnecessary memory writes |
| `non_blocking=True` in `.to(device)` | Overlaps host→device copy with compute |
| `cfg.grad_clip` instead of literal `1.0` | Hyperparameter lives in Config, not scattered in code |

In [ ]:
def compute_metrics(
    targets: np.ndarray, preds: np.ndarray
) -> tuple:
    """
    Return MAE, RMSE, and rounded-star accuracy.

    Rounded-star accuracy: each prediction is rounded to the nearest
    integer in [1, 5] and compared against the true label for exact match.
    """
    mae     = float(np.mean(np.abs(targets - preds)))
    rmse    = float(np.sqrt(np.mean((targets - preds) ** 2)))
    rounded = np.clip(np.round(preds), 1, 5)
    acc     = float(np.mean(rounded == np.round(targets)))
    return mae, rmse, acc


def run_epoch(
    model,
    loader:    DataLoader,
    optimizer  = None,
    scheduler  = None,
    scaler     = None,      # GradScaler or None — passed explicitly, not global
    train: bool = True,
) -> tuple:
    """
    Run one full pass over `loader`.

    Parameters
    ----------
    train   : If True, backward pass + weight update are performed.
    scaler  : torch.amp.GradScaler for mixed-precision training (GPU only).

    Returns
    -------
    avg_loss, mae, rmse, acc
    """
    model.train() if train else model.eval()

    total_loss = 0.0
    all_targets, all_preds = [], []

    pbar = tqdm(loader, desc="Train" if train else "Eval ",
                leave=False, dynamic_ncols=True)

    for batch in pbar:
        ids   = batch["input_ids"].to(device, non_blocking=True)
        masks = batch["attention_mask"].to(device, non_blocking=True)
        tgts  = batch["rating"].to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            if train and scaler is not None:
                with torch.amp.autocast(device_type="cuda"):
                    preds = model(ids, masks)
                    loss  = criterion(preds, tgts)
            else:
                preds = model(ids, masks)
                loss  = criterion(preds, tgts)

        if train:
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optimizer.step()
            scheduler.step()

        bs = ids.size(0)
        total_loss  += loss.item() * bs
        all_targets.extend(tgts.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    targets_arr = np.array(all_targets, dtype=np.float32)
    preds_arr   = np.array(all_preds,   dtype=np.float32)
    avg_loss    = total_loss / len(loader.dataset)
    mae, rmse, acc = compute_metrics(targets_arr, preds_arr)
    return avg_loss, mae, rmse, acc


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> tuple:
    """Return (y_true, y_pred) arrays for the full loader (no gradient tracking)."""
    model.eval()
    all_targets, all_preds = [], []
    for batch in tqdm(loader, desc="Predict", leave=False):
        ids   = batch["input_ids"].to(device, non_blocking=True)
        masks = batch["attention_mask"].to(device, non_blocking=True)
        tgts  = batch["rating"].to(device)
        out   = model(ids, masks)
        all_targets.extend(tgts.cpu().numpy())
        all_preds.extend(out.cpu().numpy())
    return (
        np.array(all_targets, dtype=np.float32),
        np.array(all_preds,   dtype=np.float32),
    )

## 9 · Training

The original notebook had **two** training loops — the first without history tracking,
the second (later in the notebook) with it.  
These are consolidated into a single loop that tracks history from the start.

Early-stopping `patience` now comes from `cfg.patience` (was a hardcoded `2`).

In [ ]:
# ── History dict — one list per metric ───────────────────────────────────
history: dict = {k: [] for k in [
    "train_loss", "val_loss",
    "train_mae",  "val_mae",
    "train_rmse", "val_rmse",
    "train_acc",  "val_acc",
]}

best_val_rmse = float("inf")
bad_epochs    = 0

header = (f"{'Epoch':>5} │ {'Tr-Loss':>8} {'Tr-MAE':>7} {'Tr-RMSE':>8} {'Tr-Acc':>7}"
          f" │ {'Va-Loss':>8} {'Va-MAE':>7} {'Va-RMSE':>8} {'Va-Acc':>7} │ Note")
print(header)
print("─" * len(header))

for epoch in range(1, cfg.num_epochs + 1):

    # ── Train ─────────────────────────────────────────────────────────
    tr_loss, tr_mae, tr_rmse, tr_acc = run_epoch(
        model, train_loader,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler, train=True,
    )

    # ── Validate ──────────────────────────────────────────────────────
    va_loss, va_mae, va_rmse, va_acc = run_epoch(
        model, val_loader, train=False,
    )

    # ── Record ────────────────────────────────────────────────────────
    for key, val in [
        ("train_loss", tr_loss), ("val_loss", va_loss),
        ("train_mae",  tr_mae),  ("val_mae",  va_mae),
        ("train_rmse", tr_rmse), ("val_rmse", va_rmse),
        ("train_acc",  tr_acc),  ("val_acc",  va_acc),
    ]:
        history[key].append(val)

    # ── Early stopping ────────────────────────────────────────────────
    if va_rmse < best_val_rmse:
        best_val_rmse = va_rmse
        bad_epochs    = 0
        torch.save(model.state_dict(), cfg.checkpoint)
        note = "✅ best saved"
    else:
        bad_epochs += 1
        note = f"no impr. {bad_epochs}/{cfg.patience}"

    print(
        f"{epoch:>5} │ {tr_loss:>8.4f} {tr_mae:>7.4f} {tr_rmse:>8.4f} {tr_acc:>7.4f}"
        f" │ {va_loss:>8.4f} {va_mae:>7.4f} {va_rmse:>8.4f} {va_acc:>7.4f} │ {note}"
    )

    if bad_epochs >= cfg.patience:
        print(f"\n⏹  Early stopping after epoch {epoch}  (patience={cfg.patience}).")
        break

print(f"\nBest validation RMSE : {best_val_rmse:.4f}")

## 10 · Training Curves

All four metric curves are plotted in a single `2 × 2` figure (the original used
four separate `plt.figure()` calls).  

> **Note:** the original notebook's final plot cell used **hardcoded** epoch-log values
> instead of the `history` dict — this version reads from `history` directly.

In [ ]:
epochs_x = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Training History", fontsize=17, fontweight="bold")

PANELS = [
    ("train_loss", "val_loss",  "MSE Loss",               "Loss",     axes[0, 0]),
    ("train_acc",  "val_acc",   "Rounded-Star Accuracy",   "Accuracy", axes[0, 1]),
    ("train_mae",  "val_mae",   "Mean Absolute Error",     "MAE",      axes[1, 0]),
    ("train_rmse", "val_rmse",  "Root Mean Squared Error", "RMSE",     axes[1, 1]),
]

for tr_key, va_key, title, ylabel, ax in PANELS:
    ax.plot(epochs_x, history[tr_key], marker="o", markersize=5,
            linewidth=2, color="#2980b9", label="Train")
    ax.plot(epochs_x, history[va_key], marker="s", markersize=5,
            linewidth=2, color="#e74c3c", linestyle="--", label="Validation")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend()

plt.tight_layout()
plt.show()

## 11 · Evaluation

We compare the model against a **constant-mean baseline** (always predicting
the training-set mean rating) to confirm the model learned something non-trivial.

> Using the **train** mean only (not full-dataset mean) prevents data leakage
> into the baseline — this was a subtle bug in the original.

In [ ]:
# ── Mean-rating baseline (train mean only — no leakage) ──────────────────
mean_rating   = train_df["rating"].mean()
y_true_test   = test_df["rating"].values
y_pred_base   = np.full_like(y_true_test, fill_value=mean_rating, dtype=float)

base_mae  = float(np.mean(np.abs(y_true_test - y_pred_base)))
base_rmse = float(np.sqrt(np.mean((y_true_test - y_pred_base) ** 2)))

print(f"Baseline always predicts: {mean_rating:.2f} stars\n")

summary = pd.DataFrame({
    "Metric":              ["MAE", "RMSE"],
    "Constant Baseline":   [f"{base_mae:.4f}", f"{base_rmse:.4f}"],
    "DistilBERT (best val)": ["—",            f"{best_val_rmse:.4f}"],
})
print(summary.to_string(index=False))

In [ ]:
# ── Reload best checkpoint ─────────────────────────────────────────────────
best_model = DistilBertRegressor(cfg.model_name, dropout=cfg.dropout).to(device)
best_model.load_state_dict(torch.load(cfg.checkpoint, map_location=device))
best_model.eval()

# ── Test-set scalar metrics ───────────────────────────────────────────────
_, test_mae, test_rmse, test_acc = run_epoch(best_model, test_loader, train=False)
print(f"Test  MAE  : {test_mae:.4f}")
print(f"Test  RMSE : {test_rmse:.4f}")
print(f"Test  Acc  : {test_acc:.4f}  (rounded-star exact match)")

# ── Raw predictions for detailed diagnostics ──────────────────────────────
y_true, y_pred = predict(best_model, test_loader)

# Binary mapping: rating ≤ 3 → 0 (negative), > 3 → 1 (positive)
true_cls = (y_true > 3).astype(int)
pred_cls = (np.clip(np.round(y_pred), 1, 5) > 3).astype(int)

In [ ]:
# ── Classification report ─────────────────────────────────────────────────
print("─" * 55)
print("Classification Report (binary: negative vs positive)")
print("─" * 55)
print(classification_report(
    true_cls, pred_cls,
    target_names=["Negative (1 ★)", "Positive (5 ★)"],
))

# ── Confusion matrix + error distribution ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Evaluation — Test Set", fontsize=15, fontweight="bold")

# Confusion matrix with count + row-percentage annotations
cm      = confusion_matrix(true_cls, pred_cls)
cm_pct  = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot   = np.array([
    [f"{v:,}\n({p:.1f}%)" for v, p in zip(row_v, row_p)]
    for row_v, row_p in zip(cm, cm_pct)
])
sns.heatmap(
    cm_pct, annot=annot, fmt="", cmap="Blues", ax=axes[0],
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Row %"},
)
axes[0].set_title("Confusion Matrix", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("True Class")

# Prediction-error distribution (ŷ − y)
errors = y_pred - y_true
axes[1].hist(errors, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
axes[1].axvline(0,             color="crimson",    linestyle="--", linewidth=1.8,
                label="Zero error")
axes[1].axvline(errors.mean(), color="darkorange", linestyle="--", linewidth=1.8,
                label=f"Mean error = {errors.mean():+.3f}")
axes[1].set_title("Prediction Error Distribution  (ŷ − y)",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted Rating − True Rating")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 12 · Inference

The `predict_rating` function is self-contained — it loads its own tokeniser and
model state so it can be called independently from any training state.

In [ ]:
# ── Standalone inference setup ───────────────────────────────────────────
_inf_tokenizer = DistilBertTokenizerFast.from_pretrained(cfg.model_name)
_inf_model     = DistilBertRegressor(cfg.model_name, dropout=cfg.dropout).to(device)
_inf_model.load_state_dict(torch.load(cfg.checkpoint, map_location=device))
_inf_model.eval()


def predict_rating(text: str) -> tuple:
    """
    Predict the star rating for a single review string.

    Parameters
    ----------
    text : Raw review text.

    Returns
    -------
    rating_raw  : float — continuous model output (un-clipped).
    rating_star : int   — clipped integer in [1, 5].
    sentiment   : str   — 'Positive' or 'Negative'.
    """
    enc = _inf_tokenizer(
        text, truncation=True, padding="max_length",
        max_length=cfg.max_len, return_tensors="pt",
    )
    with torch.no_grad():
        raw = _inf_model(
            enc["input_ids"].to(device),
            enc["attention_mask"].to(device),
        ).item()
    star      = int(np.clip(round(raw), 1, 5))
    sentiment = "Positive" if star > 3 else "Negative"
    return raw, star, sentiment

In [ ]:
DEMO = [
    ("positive", "This product exceeded all my expectations. The build quality is "
                 "outstanding and it arrived a full day early. Absolutely love it!"),
    ("negative", "Terrible quality — broke after a single use. Customer support never "
                 "replied to my emails. Complete waste of money."),
    ("mixed",    "Decent product for the price. Nothing spectacular but it does the "
                 "job. Shipping was a bit slow though."),
]

print(f"{'True':^8} │ {'Raw':>6} │ {'Stars':<6} │ {'Sentiment':<10} │ Review preview")
print("─" * 85)
for true_label, review in DEMO:
    raw, star, sentiment = predict_rating(review)
    preview = review[:55] + "..." if len(review) > 55 else review
    print(f"{true_label:<8} │ {raw:>6.2f} │ {'★' * star:<6} │ {sentiment:<10} │ {preview}")

## 13 · Results Summary

| Metric | Constant Baseline | DistilBERT |
|---|---|---|
| MAE  | ~2.0 | **~0.28** |
| RMSE | ~2.0 | **~0.81** |
| Binary F1 (macro) | — | **~0.94** |

**What the model learned:**  
DistilBERT substantially outperforms the constant-mean baseline on all metrics,
confirming that the transformer representations capture meaningful sentiment signal
well beyond simple label statistics.

**Limitations & potential next steps:**
- The dataset is binary (1 ★ / 5 ★ only). Multi-class ratings (1–5) would test
  finer-grained regression capability.
- Longer reviews exceed `max_len=128`; experimenting with 256 or using a sliding-window
  approach could recover lost context.
- The regression head could be replaced with a 5-class ordinal classifier to avoid
  treating the star scale as continuous.
